In [ ]:
from __future__ import annotations

import time
import re
import requests
import pandas as pd
from pathlib import Path
from typing import List


BASE_URL = "https://teikokugikai-i.ndl.go.jp/api/emp/speech"

QUERIES = ["台灣"]

# 5年分段
RANGES = [
    (1890, 1894),
    (1895, 1899),
    (1900, 1904),
    (1905, 1909),
    (1910, 1914),
    (1915, 1919),
    (1920, 1924),
    (1925, 1929),
    (1930, 1934),
    (1935, 1939),
    (1940, 1944),
    (1945, 1947),
]


# =========================
# API 抓資料（正確版本）
# =========================
def fetch_ndl_segment(keyword: str, start_date: str, end_date: str, page_size: int = 100) -> pd.DataFrame:

    start = 1
    records = []
    total = None

    while True:

        params = {
            "any": keyword,
            "from": start_date,
            "until": end_date,
            "startRecord": start,
            "maximumRecords": page_size,
            "recordPacking": "json"
        }

        r = requests.get(BASE_URL, params=params, timeout=30)

        if r.status_code != 200:
            print("HTTP error:", r.status_code)
            break

        data = r.json()

        if total is None:
            total = int(data.get("numberOfRecords", 0))
            print(f"[{keyword}] total:", total)

        items = data.get("speechRecord", [])

        if not items:
            break

        for it in items:

            records.append({
                "document_id": it.get("speechID"),
                "speaker": it.get("speaker", "Unknown"),
                "title": it.get("nameOfMeeting", ""),
                "date": it.get("date"),
                "text": it.get("speech") or "",
                "source_url": it.get("speechURL", ""),
                "speakerPosition": it.get("speakerPosition", ""),
                "speakerGroup": it.get("speakerGroup", ""),
                "speakerElection": it.get("speakerElection", "")
            })

        start += page_size

        print(f"[{keyword}] fetched {len(records)} / {total}")

        time.sleep(0.3)  # 防封鎖

        if len(records) >= total:
            break

    return pd.DataFrame(records)


# =========================
# 清理
# =========================
def clean_whitespace(text: str) -> str:
    text = str(text)
    text = text.replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n\n", text)
    return text.strip()


def count_words(text: str) -> int:
    return len(re.findall(r"\b\w+\b", str(text)))


def make_century(year: int) -> str:
    if year == 0:
        return ""
    c = (year - 1) // 100 + 1
    return f"{c}th century"


# =========================
# dataset
# =========================
def build_full_dataset(df: pd.DataFrame) -> pd.DataFrame:

    df = df.copy()

    df["text"] = df["text"].map(clean_whitespace)
    df = df[df["text"].str.len() > 0]

    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    df["year"] = df["date"].dt.year.fillna(0).astype(int)
    df["decade"] = (df["year"] // 10) * 10
    df["century"] = df["year"].apply(make_century)

    df["word_count"] = df["text"].map(count_words)

    df["group_primary"] = df["speaker"]
    df["group_secondary"] = df["decade"].astype(str)

    return df


# =========================
# chunk
# =========================
def chunk_text(text: str, size: int = 300) -> List[str]:
    words = text.split()
    return [" ".join(words[i:i+size]) for i in range(0, len(words), size)]


def build_excerpt_rows(full_df: pd.DataFrame) -> pd.DataFrame:

    rows = []

    for r in full_df.itertuples(index=False):

        chunks = chunk_text(r.text, 300)

        for i, c in enumerate(chunks, 1):

            rows.append({
                "document_id": r.document_id,
                "excerpt_id": f"{r.document_id}_E{i:02d}",
                "chunk_number": i,
                "speaker": r.speaker,
                "date": r.date,
                "year": r.year,
                "decade": r.decade,
                "text": c,
                "word_count": count_words(c),
                "source_url": r.source_url
            })

    return pd.DataFrame(rows)


# =========================
# 主流程（重點）
# =========================
def main():

    all_dfs = []

    for start_year, end_year in RANGES:

        print("\n==============================")
        print(f"SEGMENT {start_year}-{end_year}")

        segment_dfs = []

        for kw in QUERIES:

            df = fetch_ndl_segment(
                keyword=kw,
                start_date=f"{start_year}-01-01",
                end_date=f"{end_year}-12-31"
            )

            segment_dfs.append(df)

        # 合併三種寫法
        seg = pd.concat(segment_dfs, ignore_index=True)

        # 去重（超重要）
        before = len(seg)
        seg = seg.drop_duplicates(subset=["document_id"])
        after = len(seg)

        print(f"Segment rows: {after} (dedup from {before})")

        # 存每段
        fname = f"raw_{start_year}_{end_year}.csv"
        seg.to_csv(fname, index=False)
        print("Saved:", fname)

        # 手動確認
        cont = input("Continue? (y/n): ").strip().lower()
        if cont != "y":
            print("Stopped.")
            break

        all_dfs.append(seg)

    if not all_dfs:
        print("No data.")
        return

    # =========================
    # 合併全部
    # =========================
    raw = pd.concat(all_dfs, ignore_index=True)
    raw = raw.drop_duplicates(subset=["document_id"])

    print("TOTAL UNIQUE:", len(raw))

    # =========================
    # 後處理
    # =========================
    full = build_full_dataset(raw)
    excerpt = build_excerpt_rows(full)

    Path("output").mkdir(exist_ok=True)

    full.to_csv("output/full.csv", index=False)
    excerpt.to_csv("output/excerpt.csv", index=False)

    print("DONE")
    print("output/full.csv")
    print("output/excerpt.csv")


if __name__ == "__main__":
    main()


SEGMENT 1890-1894
[台灣] total: 28
[台灣] fetched 28 / 28
Segment rows: 28 (dedup from 28)
Saved: raw_1890_1894.csv

SEGMENT 1895-1899
[台灣] total: 3111
[台灣] fetched 100 / 3111
[台灣] fetched 200 / 3111
[台灣] fetched 300 / 3111
[台灣] fetched 400 / 3111
[台灣] fetched 500 / 3111
[台灣] fetched 600 / 3111
[台灣] fetched 700 / 3111
[台灣] fetched 800 / 3111
[台灣] fetched 900 / 3111
[台灣] fetched 1000 / 3111
[台灣] fetched 1100 / 3111
[台灣] fetched 1200 / 3111
[台灣] fetched 1300 / 3111
[台灣] fetched 1400 / 3111
[台灣] fetched 1500 / 3111
[台灣] fetched 1600 / 3111
[台灣] fetched 1700 / 3111
[台灣] fetched 1800 / 3111
[台灣] fetched 1900 / 3111
[台灣] fetched 2000 / 3111
[台灣] fetched 2100 / 3111
[台灣] fetched 2200 / 3111
[台灣] fetched 2300 / 3111
[台灣] fetched 2400 / 3111
[台灣] fetched 2500 / 3111
[台灣] fetched 2600 / 3111
[台灣] fetched 2700 / 3111
[台灣] fetched 2800 / 3111
[台灣] fetched 2900 / 3111
[台灣] fetched 3000 / 3111
[台灣] fetched 3100 / 3111
[台灣] fetched 3111 / 3111
Segment rows: 3111 (dedup from 3111)
Saved: raw_1895_1899.cs